## Global configuration

In [ ]:
import os
import torch
import torch.optim as optim
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset
import torch.optim as optim
from sklearn.metrics import accuracy_score,confusion_matrix , precision_recall_fscore_support
import os
from tqdm import tqdm
import pandas as pd
import nibabel as nib
import numpy as np
import logging
import torch
from sklearn.preprocessing import StandardScaler
from tqdm.notebook import tqdm
import time
import psutil
import gc

In [ ]:
class GlobalConfig:
    def __init__(self):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.base_dir = '/kaggle/input/datasets/tolbaadel/bmd3dataset'
        self.dataset_dir = os.path.join(self.base_dir, 'Cropped_NIfTI_merged/Cropped_NIfTI_merged')
        self.csv_path = os.path.join(self.base_dir, 'paired_data_datscan_augmented.csv')
        self.results_dir = './results'
        self.models_dir = './models'
        self.graphs_dir = './graphs'
        self.setup_environment()
    
    def setup_environment(self):
        print(f"🚀 Using device: {self.device}")
        print(f"📁 Dataset directory: {self.dataset_dir}")
        print(f"📁 CSV file: {self.csv_path}")
        
        if os.path.exists(self.dataset_dir):
            print(f"✅ Dataset folder found")
        else:
            print(f"❌ Dataset folder NOT found")
        
        if os.path.exists(self.csv_path):
            print(f"✅ CSV file found")
        else:
            print(f"❌ CSV file NOT found")
        
        for d in [self.results_dir, self.models_dir, self.graphs_dir]:
            if not os.path.exists(d):
                os.makedirs(d)
                print(f"📁 Created folder: {d}")
            else:
                print(f"✅ Folder exists: {d}")

CONFIG = GlobalConfig()

🚀 Using device: cuda
📁 Dataset directory: /kaggle/input/bmd3dataset/bm3d_version/Cropped_NIfTI_merged
📁 CSV file: /kaggle/input/bmd3dataset/bm3d_version/paired_data_datscan_augmented.csv
✅ Dataset folder found
✅ CSV file found
✅ Folder exists: ./results
✅ Folder exists: ./models
✅ Folder exists: ./graphs


## Data loading

In [21]:
class DaTscanDataset(Dataset):
    """Self-contained dataset using image_id for pairing with auto-validation"""
    def __init__(self, datscan_volumes, medical_records, labels, image_ids, transform=None):
        self.transform = transform
        # Store data as a dictionary with image_id as key
        if image_ids is None or len(image_ids) == 0:
            raise ValueError("image_ids must be provided and non-empty")
        self.data = {
            image_id: {
                'volume': vol,
                'medical': med,
                'label': lbl
            }
            for image_id, vol, med, lbl in zip(image_ids, datscan_volumes, medical_records, labels)
        }
        self.image_ids = list(self.data.keys())  # List of image_ids for indexing during iteration

        # Auto-validate data
        self._validate_data()

        print(f"📊 Dataset created: {len(self)} samples")
        print(f"   DaTscan shape: {self.data[self.image_ids[0]]['volume'].shape if len(self.data) > 0 else 'Empty'}")
        print(f"   Medical features: {self.data[self.image_ids[0]]['medical'].shape[0] if len(self.data) > 0 and hasattr(self.data[self.image_ids[0]]['medical'], 'shape') else 'N/A'}")
        print(f"   Label distribution: {np.bincount([self.data[id]['label'] for id in self.image_ids]) if len(self.data) > 0 else 'Empty'}")

    def _validate_data(self):
        """Auto-validate dataset"""
        # Check if data has the same length , matches the nbr of data enteries.
        num_samples = len(self.image_ids)
        assert num_samples == len(self.data), \
            f"Data length mismatch! Image IDs: {num_samples}, Data entries: {len(self.data)}"

        # Verify each image_id has valid data
        for image_id in self.image_ids:
            if image_id not in self.data:
                raise KeyError(f"Image ID {image_id} not found in data dictionary")
            entry = self.data[image_id]
            assert 'volume' in entry and 'medical' in entry and 'label' in entry, \
                f"Incomplete data for Image ID {image_id}"

        if num_samples > 0:  # Confirm the image dimensions
            actual_shape = self.data[self.image_ids[0]]['volume'].shape
            print(f"✅ DaTscan shape: {actual_shape}")

    def __len__(self):
        """Tells DataLoader how many items are in the dataset"""
        return len(self.image_ids)

    def __getitem__(self, idx):
        """Prep data for training using image_id"""
        image_id = self.image_ids[idx]
        entry = self.data[image_id]

        volume = entry['volume']
        if isinstance(volume, np.ndarray):
            volume = torch.from_numpy(volume).float()

        if volume.dim() == 3:
            volume = volume.unsqueeze(0)

        medical = entry['medical']
        if isinstance(medical, np.ndarray):
            medical = torch.from_numpy(medical).float()

        label = entry['label']
        if isinstance(label, np.ndarray):
            label = torch.from_numpy(label).long()
        elif not isinstance(label, torch.Tensor):
            label = torch.tensor(label, dtype=torch.long)

        if self.transform:
            volume = self.transform(volume)

        return volume, medical, label, image_id  # Return image_id for traceability


In [22]:
class RealDataLoader:
    """Load and pair real CSV and NIfTI data"""
    def __init__(self, csv_path, nifti_dir):
        self.csv_path = csv_path  # Path to CSV file
        self.nifti_dir = nifti_dir  # Path to NIfTI directory
        self.scaler = StandardScaler()  # For normalizing medical features
        self.num_features = None  # Store number of features
        logging.info(f"Initialized RealDataLoader with CSV: {csv_path}, NIfTI dir: {nifti_dir}")

    def load_and_preprocess_data(self):
        """Load CSV and NIfTI data, pair them, and preprocess"""
        try:
            logging.info("🔄 Loading real data...")

            # Load CSV data
            logging.info(f"📊 Loading CSV from: {self.csv_path}")
            if not os.path.exists(self.csv_path):
                raise FileNotFoundError(f"CSV file not found: {self.csv_path}")
            df = pd.read_csv(self.csv_path)
            logging.info(f"   CSV shape: {df.shape}")

            # Clean the CSV data
            df = self._clean_csv_data(df)

            # Load and pair images
            logging.info(f"🧠 Loading NIfTI images from: {self.nifti_dir}")
            paired_data = self._pair_csv_with_images(df)

            if len(paired_data) == 0:
                raise ValueError("No matching image-CSV pairs found!")

            logging.info(f"✅ Successfully paired {len(paired_data)} samples")

            # ---- NEW: Print summary table ----
            label_counts = pd.Series([d['csv_row']['NHY'] for d in paired_data]).value_counts().sort_index()
            summary_df = pd.DataFrame({
                "Class": label_counts.index,
                "Samples": label_counts.values
            })
            print("\n📊 Pairing Summary (Images ↔ CSV):")
            print(summary_df.to_string(index=False))
            print(f"Total paired: {len(paired_data)} out of {len(df)} CSV rows\n")

            # Extract features, labels, and image IDs
            volumes, medical_records, labels, image_ids = self._extract_features_labels(paired_data)

            # Normalize medical features
            medical_records = self.scaler.fit_transform(medical_records)
            logging.info(f"   Normalized medical features: {medical_records.shape}")

            return volumes, medical_records, labels, image_ids

        except Exception as e:
            logging.error(f"❌ Data loading failed: {e}")
            return None, None, None, None

    def _clean_csv_data(self, df):
        """Clean and preprocess CSV data"""
        logging.info("🧹 Cleaning CSV data...")

        # Drop rows with missing Image Data ID
        initial_rows = len(df)
        if 'Image Data ID' not in df.columns:
            raise KeyError("CSV missing 'Image Data ID' column")
        df = df.dropna(subset=['Image Data ID'])
        logging.info(f"   Dropped {initial_rows - len(df)} rows with missing Image Data ID")

        # Drop rows with missing NHY (target variable)
        if 'NHY' not in df.columns:
            raise KeyError("CSV missing 'NHY' column")
        df = df.dropna(subset=['NHY'])
        logging.info(f"   Rows after dropping NaN labels: {len(df)}")

        # Convert NHY to integer (severity stages)
        df['NHY'] = df['NHY'].astype(int)

        # Handle missing values in numeric features
        feature_columns = df.select_dtypes(include=[np.number]).columns.tolist()
        feature_columns = [col for col in feature_columns if col not in ['PATNO', 'NHY']]
        for col in feature_columns:
            if df[col].isnull().sum() > 0:
                median_val = df[col].median()
                df[col].fillna(median_val, inplace=True)
                logging.info(f"   Filled missing values in {col} with median: {median_val}")

        logging.info(f"   Final cleaned CSV shape: {df.shape}")
        return df

    def _pair_csv_with_images(self, df):
        """Pair CSV rows with corresponding NIfTI images"""
        paired_data = []

        # Build dictionary of all available images
        image_files = {}
        if not os.path.exists(self.nifti_dir):
            raise FileNotFoundError(f"NIfTI directory not found: {self.nifti_dir}")
        patient_folders = os.listdir(self.nifti_dir)
        logging.info(f"   Found {len(patient_folders)} patient folders")

        # Search through patient folders
        for folder in patient_folders:
            folder_path = os.path.join(self.nifti_dir, folder)
            if os.path.isdir(folder_path):
                try:
                    files_in_folder = os.listdir(folder_path)
                    for file in files_in_folder:
                        if file.endswith(('.nii', '.nii.gz')):
                            image_files[file] = os.path.join(folder_path, file)
                            base_name = file.replace('.nii.gz', '').replace('.nii', '')
                            image_files[base_name] = os.path.join(folder_path, file)
                except Exception as e:
                    logging.warning(f"   Could not read folder {folder}: {e}")

        logging.info(f"   Total images found: {len(image_files)}")
        logging.info(f"   Sample image keys: {list(image_files.keys())[:5]}")

        matched_count = 0
        for _, row in df.iterrows():
            image_id = str(row['Image Data ID']).strip()

            # Try multiple naming patterns
            possible_keys = [
                image_id,
                f"I{image_id}",
                f"{image_id}.nii",
                f"{image_id}.nii.gz",
                f"I{image_id}.nii",
                f"I{image_id}.nii.gz"
            ]

            image_path = None
            for key in possible_keys:
                if key in image_files:
                    image_path = image_files[key]
                    break

            if image_path and os.path.exists(image_path):
                try:
                    # Load NIfTI image
                    nii_img = nib.load(image_path)
                    volume = nii_img.get_fdata()
                    # Normalize volume
                    volume = (volume - volume.mean()) / (volume.std() + 1e-8)

                    paired_data.append({
                        'volume': volume,
                        'csv_row': row,
                        'image_id': image_id,
                        'image_path': image_path
                    })
                    matched_count += 1
                    if matched_count % 50 == 0:
                        logging.info(f"   Matched {matched_count} images so far...")

                except Exception as e:
                    logging.warning(f"   Failed to load {image_path}: {e}")

        logging.info(f"   Successfully matched {matched_count} out of {len(df)} CSV rows")
        return paired_data

    def _extract_features_labels(self, paired_data):
        """Extract volumes, medical features, labels, and image IDs from paired data"""
        volumes = []
        medical_records = []
        labels = []
        image_ids = []

        # Get feature columns (exclude identifiers and target)
        exclude_cols = ['PATNO', 'EVENT_ID', 'Image Data ID', 'NHY']
        sample_row = paired_data[0]['csv_row']
        feature_columns = [col for col in sample_row.index if col not in exclude_cols]
        self.num_features = len(feature_columns)  # Store in instance variable
        logging.info(f"📊 Using {self.num_features} medical features")
        logging.info(f"   Feature columns: {feature_columns[:5]}...")

        for data in paired_data:
            volumes.append(data['volume'])
            medical_features = data['csv_row'][feature_columns].values.astype(np.float32)
            medical_records.append(medical_features)
            labels.append(data['csv_row']['NHY'])
            image_ids.append(data['image_id'])

        return np.array(volumes), np.array(medical_records), np.array(labels), image_ids

    def get_num_features(self):
        """Get the number of medical features"""
        return self.num_features

In [23]:
def _auto_create_real_dataset(config, num_folds=5, checkpoint_path=None, num_features_target=81):
    """
    Create train/val/test DataLoaders and full dataset from real data with cross-validation support.
    """
    try:
        # Use the correct CSV and NIfTI paths from config
        csv_path = config.csv_path  # e.g. /kaggle/input/threshhold-dataset/Thershhold-Dataset/assesments_data.csv
        nifti_dir = config.dataset_dir  # e.g. /kaggle/input/threshhold-dataset/Thershhold-Dataset/Processed_NIfTI

        logging.info(f"📁 CSV path: {csv_path}")
        logging.info(f"📁 NIfTI path: {nifti_dir}")

        # Check file existence
        if not os.path.exists(csv_path):
            raise FileNotFoundError(f"CSV file not found: {csv_path}")
        if not os.path.exists(nifti_dir):
            raise FileNotFoundError(f"NIfTI directory not found: {nifti_dir}")

        # Load and preprocess data
        data_loader = RealDataLoader(csv_path, nifti_dir)
        volumes, medical_records, labels, image_ids = data_loader.load_and_preprocess_data()

        if volumes is None:
            raise ValueError("Data loading failed, no data returned")

        # Get number of features from the data loader
        num_features = data_loader.get_num_features()

        # Validate data consistency
        if len(image_ids) != len(volumes) or len(volumes) != len(medical_records) or len(medical_records) != len(labels):
            raise ValueError(f"Data mismatch: {len(image_ids)} IDs, {len(volumes)} volumes, {len(medical_records)} records, {len(labels)} labels")

        # Ensure exactly 81 medical features
        if num_features != num_features_target:
            if num_features < num_features_target:
                raise ValueError(f"Insufficient features: {num_features} available, {num_features_target} required")
            logging.warning(f"Reducing medical features from {num_features} to {num_features_target}")
            medical_records = medical_records[:, :num_features_target]
            num_features = num_features_target

        logging.info(f"✅ Loaded {len(volumes)} samples with {num_features} medical features")
        label_counts = np.bincount(labels)
        logging.info(f"   Label distribution: {label_counts}")

        # Check for classes with insufficient samples for stratified k-fold
        min_samples_for_kfold = num_folds
        small_classes = np.where(label_counts < min_samples_for_kfold)[0]
        use_stratified = len(small_classes) == 0

        if not use_stratified:
            logging.warning(f"⚠️ Classes with insufficient samples for {num_folds}-fold CV: {dict(zip(small_classes, label_counts[small_classes]))}")
            logging.warning(f"   Need at least {min_samples_for_kfold} samples per class for stratified {num_folds}-fold CV")
            logging.warning("   Using non-stratified splitting")

        # Create full dataset
        transform = None
        full_dataset = DaTscanDataset(volumes, medical_records, labels, image_ids, transform=transform)
        logging.info(f"Created full dataset with {len(full_dataset)} samples")

        # Cross-validation splits
        from sklearn.model_selection import KFold, StratifiedKFold
        if use_stratified:
            try:
                skf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)
                cv_splits = [(train_idx, val_idx) for train_idx, val_idx in skf.split(np.zeros(len(labels)), labels)]
                logging.info(f"✅ Prepared {num_folds}-fold stratified cross-validation splits")
            except ValueError as e:
                logging.warning(f"Stratified CV failed: {e}")
                logging.warning("Falling back to regular KFold")
                kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)
                cv_splits = [(train_idx, val_idx) for train_idx, val_idx in kf.split(np.zeros(len(labels)))]
                logging.info(f"✅ Prepared {num_folds}-fold regular cross-validation splits")
        else:
            kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)
            cv_splits = [(train_idx, val_idx) for train_idx, val_idx in kf.split(np.zeros(len(labels)))]
            logging.info(f"✅ Prepared {num_folds}-fold regular cross-validation splits")

        # Train/val/test split
        from sklearn.model_selection import train_test_split
        if use_stratified:
            try:
                train_indices, temp_indices = train_test_split(
                    range(len(full_dataset)), test_size=0.3, random_state=42, stratify=labels
                )
                temp_labels = labels[temp_indices]
                temp_counts = np.bincount(temp_labels)
                if np.min(temp_counts[temp_counts > 0]) >= 2:
                    val_indices, test_indices = train_test_split(
                        temp_indices, test_size=0.5, random_state=42, stratify=temp_labels
                    )
                    logging.info("✅ Used stratified train/val/test split")
                else:
                    val_indices, test_indices = train_test_split(
                        temp_indices, test_size=0.5, random_state=42
                    )
                    logging.info("✅ Used stratified train/temp split, then random val/test split")
            except ValueError as e:
                logging.warning(f"Stratified split failed: {e}")
                logging.warning("Using random split instead")
                train_indices, temp_indices = train_test_split(
                    range(len(full_dataset)), test_size=0.3, random_state=42
                )
                val_indices, test_indices = train_test_split(
                    temp_indices, test_size=0.5, random_state=42
                )
                logging.info("✅ Used random train/val/test split")
        else:
            train_indices, temp_indices = train_test_split(
                range(len(full_dataset)), test_size=0.3, random_state=42
            )
            val_indices, test_indices = train_test_split(
                temp_indices, test_size=0.5, random_state=42
            )
            logging.info("✅ Used random train/val/test split")

        # Create datasets
        train_dataset = DaTscanDataset(
            volumes[train_indices], medical_records[train_indices], labels[train_indices],
            [image_ids[i] for i in train_indices], transform=transform
        )
        val_dataset = DaTscanDataset(
            volumes[val_indices], medical_records[val_indices], labels[val_indices],
            [image_ids[i] for i in val_indices], transform=None
        )
        test_dataset = DaTscanDataset(
            volumes[test_indices], medical_records[test_indices], labels[test_indices],
            [image_ids[i] for i in test_indices], transform=None
        )

        # Create DataLoaders
        from torch.utils.data import DataLoader
        train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0)
        val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=0)
        test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=0)

        logging.info(f"✅ DataLoaders created:")
        logging.info(f"   Training: {len(train_dataset)} samples (70%)")
        logging.info(f"   Validation: {len(val_dataset)} samples (15%)")
        logging.info(f"   Testing: {len(test_dataset)} samples (15%)")

        # Check checkpoint
        if checkpoint_path and os.path.exists(checkpoint_path):
            logging.info(f"Checkpoint found at: {checkpoint_path}")

        # Return full dataset and indices for CV flexibility
        all_indices = {'train': train_indices, 'val': val_indices, 'test': test_indices}
        return train_loader, val_loader, test_loader, num_features_target, cv_splits, full_dataset, all_indices

    except Exception as e:
        print(f"Error in dataset creation: {e}")
        logging.error(f"❌ Dataset creation failed: {e}")
        return None, None, None, None, None, None, None


In [24]:
# Call the function
train_loader, val_loader, test_loader, num_features, cv_splits, full_dataset, all_indices = _auto_create_real_dataset(
    config=CONFIG, num_folds=5, checkpoint_path=None, num_features_target=81
)

# Assign to global variables expected by auto_detect_real_data
global REAL_TRAIN_LOADER, REAL_TEST_LOADER, FULL_DATASET, ALL_INDICES
REAL_TRAIN_LOADER = train_loader  # Use train_loader for training
REAL_TEST_LOADER = test_loader    # Use test_loader for validation/testing
FULL_DATASET = full_dataset       # Store full dataset
ALL_INDICES = all_indices         # Store indices for CV

# Verify output
if train_loader is not None:
    print(f"Loaded: {len(train_loader.dataset)} train, {len(val_loader.dataset)} val, {len(test_loader.dataset)} test samples")
    print(f"Features: {num_features}")
    print(f"CV Splits: {len(cv_splits)} folds")
    print(f"Full dataset: {len(full_dataset)} samples")
else:
    print("Dataset creation failed")


📊 Pairing Summary (Images ↔ CSV):
 Class  Samples
     0      188
     1      376
     2      753
     3      376
     4      188
Total paired: 1881 out of 1881 CSV rows

✅ DaTscan shape: (46, 54, 14)
📊 Dataset created: 1881 samples
   DaTscan shape: (46, 54, 14)
   Medical features: 81
   Label distribution: [188 376 753 376 188]
✅ DaTscan shape: (46, 54, 14)
📊 Dataset created: 1316 samples
   DaTscan shape: (46, 54, 14)
   Medical features: 81
   Label distribution: [131 263 527 263 132]
✅ DaTscan shape: (46, 54, 14)
📊 Dataset created: 282 samples
   DaTscan shape: (46, 54, 14)
   Medical features: 81
   Label distribution: [ 29  56 113  56  28]
✅ DaTscan shape: (46, 54, 14)
📊 Dataset created: 283 samples
   DaTscan shape: (46, 54, 14)
   Medical features: 81
   Label distribution: [ 28  57 113  57  28]
Loaded: 1316 train, 282 val, 283 test samples
Features: 81
CV Splits: 5 folds
Full dataset: 1881 samples


## Pre-trained models

In [ ]:
# CONFIGURATION
class ModelComparisonConfig:
    def __init__(self):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.num_epochs = 30
        self.batch_size = 8
        self.learning_rate = 1e-4
        self.weight_decay = 0.01
        self.patience = 15
        self.num_classes = 5
        self.models_dir = './models'
        self.results_dir = './results'
        self.graphs_dir = './graphs'
        for d in [self.models_dir, self.results_dir, self.graphs_dir]:
            os.makedirs(d, exist_ok=True)

COMP_CONFIG = ModelComparisonConfig()


# DATASET CLASS
class DaTscanDataset(Dataset):
    def __init__(self, data_df, transform=None):
        self.data = data_df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        volume_path = row['path']
        volume = self.load_volume(volume_path)
        if self.transform:
            volume = self.transform(volume)
        volume = torch.FloatTensor(volume).unsqueeze(0)
        label = int(row['stage'])
        medical = torch.zeros(5)
        return volume, medical, label, volume_path

    def load_volume(self, path):
        try:
            img = nib.load(str(path))
            volume = img.get_fdata()
            if volume.max() > volume.min():
                volume = (volume - volume.min()) / (volume.max() - volume.min())
            else:
                volume = np.zeros_like(volume)
            return volume.astype(np.float32)
        except Exception as e:
            print(f"Error loading {path}: {e}")
            return np.zeros((46, 54, 14), dtype=np.float32)


# === MobileNet ===
def _make_divisible(v, divisor=8, min_value=None):
    """Round channels to the nearest multiple of divisor"""
    if min_value is None:
        min_value = divisor
    new_v = max(min_value, int(v + divisor / 2) // divisor * divisor)
    # ensure rounding down does not go more than 10% below original value
    if new_v < 0.9 * v:
        new_v += divisor
    return new_v

class InvertedResidual3D(nn.Module):
    """
    MobileNetV2 inverted residual block extended to 3D with asymmetric stride:
    when stride > 1, XY dims are downsampled (stride_xy=2) but Z is preserved
    (stride_z=1) to avoid collapsing the already-thin depth dimension.
    """
    def __init__(self, in_channels, out_channels, stride, expand_ratio):
        super().__init__()
        hidden_dim = int(round(in_channels * expand_ratio))
        self.use_res_connect = stride == 1 and in_channels == out_channels
        # asymmetric 3D stride: downsample XY only, keep Z intact
        dw_stride  = (stride, stride, 1) if stride > 1 else 1
        dw_padding = (1, 1, 1)
        layers = []
        if expand_ratio != 1:
            layers += [
                nn.Conv3d(in_channels, hidden_dim, 1, 1, 0, bias=False),
                nn.BatchNorm3d(hidden_dim),
                nn.ReLU6(inplace=True),
            ]
        layers += [
            # depthwise conv with asymmetric stride
            nn.Conv3d(
                hidden_dim, hidden_dim, 3, dw_stride, dw_padding,
                groups=hidden_dim, bias=False
            ),
            nn.BatchNorm3d(hidden_dim),
            nn.ReLU6(inplace=True),
            # pointwise projection
            nn.Conv3d(hidden_dim, out_channels, 1, 1, 0, bias=False),
            nn.BatchNorm3d(out_channels),
        ]
        self.conv = nn.Sequential(*layers)

    def forward(self, x):
        if self.use_res_connect:
            return x + self.conv(x)
        else:
            return self.conv(x)

class MobileNet_3D(nn.Module):
    def __init__(self, input_shape=(46, 54, 14), num_classes=COMP_CONFIG.num_classes, width_mult=1.0):
        super().__init__()
        # channel rounding to multiples of 8
        def _c(ch):
            return _make_divisible(ch * width_mult, divisor=8)

        first_ch = _c(32)
        self.conv1 = nn.Sequential(
            # asymmetric initial stride: XY=2, Z=1
            nn.Conv3d(1, first_ch, kernel_size=3, stride=(2, 2, 1), padding=1, bias=False),
            nn.BatchNorm3d(first_ch),
            nn.ReLU6(inplace=True)
        )
        # MobileNetV2 stages
        # stages 5-7 use stride=1 because the XY dims are already small at that depth
        inverted_residual_setting = [
            # t,   c,   n, s
            [1,   16,  1, 1],
            [6,   24,  2, 2],
            [6,   32,  3, 2],
            [6,   64,  4, 2],
            [6,   96,  3, 1],
            [6,  160,  3, 1],
            [6,  320,  1, 1],
        ]
        feature_layers = []
        in_channels = first_ch
        block_idx = 0
        for stage_idx, (t, c, n, s) in enumerate(inverted_residual_setting):
            out_channels = _c(c)
            for i in range(n):
                stride = s if i == 0 else 1
                feature_layers.append(InvertedResidual3D(in_channels, out_channels, stride, t))
                in_channels = out_channels
                block_idx += 1
            # spatial dropout after stages 3 and 5 for mid-network regularisation
            if stage_idx in (3, 5):
                feature_layers.append(nn.Dropout3d(p=0.1))
        self.features = nn.Sequential(*feature_layers)
        last_ch = _make_divisible(1280 * max(1.0, width_mult), divisor=8)
        self.conv2 = nn.Sequential(
            nn.Conv3d(in_channels, last_ch, kernel_size=1, bias=False),
            nn.BatchNorm3d(last_ch),
            nn.ReLU6(inplace=True)
        )
        self.avgpool = nn.AdaptiveAvgPool3d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(last_ch, num_classes)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm3d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                nn.init.zeros_(m.bias)

    def forward(self, x, classify=True):
        x = self.conv1(x)
        x = self.features(x)
        x = self.conv2(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


# === ResNet-18 ===
class BasicBlock3D(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv3d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm3d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv3d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm3d(out_channels)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        out = self.relu(out)
        return out

class ResNet3D_18(nn.Module):
    def __init__(self, input_shape=(46, 54, 14), num_classes=COMP_CONFIG.num_classes):
        super().__init__()
        self.in_channels = 64
        # asymmetric kernel/stride: XY downsampled (7×7, s=2), Z preserved (k=3, s=1)
        self.conv1 = nn.Conv3d(
            1, 64, kernel_size=(7, 7, 3), stride=(2, 2, 1), padding=(3, 3, 1), bias=False
        )
        self.bn1 = nn.BatchNorm3d(64)
        self.relu = nn.ReLU(inplace=True)
        # asymmetric maxpool: preserve Z completely
        self.maxpool = nn.MaxPool3d(
            kernel_size=(3, 3, 1), stride=(2, 2, 1), padding=(1, 1, 0)
        )
        self.layer1 = self._make_layer(BasicBlock3D, 64, 2, stride=1)
        self.layer2 = self._make_layer(BasicBlock3D, 128, 2, stride=(2, 2, 1))
        self.layer3 = self._make_layer(BasicBlock3D, 256, 2, stride=(2, 2, 1))
        self.layer4 = self._make_layer(BasicBlock3D, 512, 2, stride=(2, 2, 1))
        self.avgpool = nn.AdaptiveAvgPool3d(1)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(512 * BasicBlock3D.expansion, num_classes)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm3d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                nn.init.zeros_(m.bias)

    def _make_layer(self, block, out_channels, blocks, stride=1):
        downsample = None
        if stride != 1 or self.in_channels != out_channels * block.expansion:
            downsample = nn.Sequential(
                nn.Conv3d(self.in_channels, out_channels * block.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm3d(out_channels * block.expansion),
            )
        layers = []
        layers.append(block(self.in_channels, out_channels, stride, downsample))
        self.in_channels = out_channels * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.in_channels, out_channels))
        return nn.Sequential(*layers)

    def forward(self, x, classify=True):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        return x


# === VGG-16 ===
class VGG3D_16(nn.Module):
    def __init__(self, input_shape=(46, 54, 14), num_classes=COMP_CONFIG.num_classes):
        super().__init__()
        self.features = nn.Sequential(
            # block 1
            nn.Conv3d(1, 64, kernel_size=3, padding=1), nn.BatchNorm3d(64), nn.ReLU(inplace=True),
            nn.Conv3d(64, 64, kernel_size=3, padding=1), nn.BatchNorm3d(64), nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=2, stride=2),
            # block 2
            nn.Conv3d(64, 128, kernel_size=3, padding=1), nn.BatchNorm3d(128), nn.ReLU(inplace=True),
            nn.Conv3d(128, 128, kernel_size=3, padding=1), nn.BatchNorm3d(128), nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=2, stride=2),
            # block 3
            nn.Conv3d(128, 256, kernel_size=3, padding=1), nn.BatchNorm3d(256), nn.ReLU(inplace=True),
            nn.Conv3d(256, 256, kernel_size=3, padding=1), nn.BatchNorm3d(256), nn.ReLU(inplace=True),
            nn.Conv3d(256, 256, kernel_size=3, padding=1), nn.BatchNorm3d(256), nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=2, stride=2),
            # block 4
            nn.Conv3d(256, 512, kernel_size=3, padding=1), nn.BatchNorm3d(512), nn.ReLU(inplace=True),
            nn.Conv3d(512, 512, kernel_size=3, padding=1), nn.BatchNorm3d(512), nn.ReLU(inplace=True),
            nn.Conv3d(512, 512, kernel_size=3, padding=1), nn.BatchNorm3d(512), nn.ReLU(inplace=True),
            # block 5
            nn.Conv3d(512, 512, kernel_size=3, padding=1), nn.BatchNorm3d(512), nn.ReLU(inplace=True),
            nn.Conv3d(512, 512, kernel_size=3, padding=1), nn.BatchNorm3d(512), nn.ReLU(inplace=True),
            nn.Conv3d(512, 512, kernel_size=3, padding=1), nn.BatchNorm3d(512), nn.ReLU(inplace=True),
        )
        self.avgpool = nn.AdaptiveAvgPool3d(1)
        self.classifier = nn.Sequential(
            nn.Linear(512, 4096), nn.ReLU(inplace=True), nn.Dropout(0.2),
            nn.Linear(4096, 4096), nn.ReLU(inplace=True), nn.Dropout(0.2),
            nn.Linear(4096, num_classes)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm3d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                nn.init.zeros_(m.bias)

    def forward(self, x, classify=True):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


# === EfficientNet3D-B0 ===
class MBConv3D(nn.Module):
    """Mobile Inverted Bottleneck Conv inflated to 3D with SE"""
    def __init__(self, in_ch, out_ch, expand_ratio, kernel_size, stride, se_ratio=0.25):
        super().__init__()
        mid_ch = in_ch * expand_ratio
        self.use_skip = (in_ch == out_ch) and (stride == 1)
        dw_stride = (stride, stride, 1) if stride > 1 else 1
        pad = kernel_size // 2
        # expand phase
        self.expand = nn.Sequential(
            nn.Conv3d(in_ch, mid_ch, 1, bias=False),
            nn.BatchNorm3d(mid_ch),
            nn.SiLU(inplace=True),
        ) if expand_ratio != 1 else nn.Identity()
        # depthwise conv: asymmetric (k,k,3) kernel to preserve Z
        self.dw = nn.Sequential(
            nn.Conv3d(
                mid_ch, mid_ch, (kernel_size, kernel_size, 3), stride=dw_stride,
                padding=(pad, pad, 1), groups=mid_ch, bias=False
            ),
            nn.BatchNorm3d(mid_ch),
            nn.SiLU(inplace=True),
        )
        # squeeze-and-excitation
        se_ch = max(1, int(in_ch * se_ratio))
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool3d(1),
            nn.Conv3d(mid_ch, se_ch, 1, bias=True),
            nn.SiLU(inplace=True),
            nn.Conv3d(se_ch, mid_ch, 1, bias=True),
            nn.Sigmoid(),
        )
        # pointwise projection
        self.project = nn.Sequential(
            nn.Conv3d(mid_ch, out_ch, 1, bias=False),
            nn.BatchNorm3d(out_ch),
        )

    def forward(self, x):
        identity = x
        out = self.expand(x)
        out = self.dw(out)
        out = out * self.se(out)
        out = self.project(out)
        if self.use_skip:
            out = out + identity
        return out

class EfficientNet3D_B0(nn.Module):
    """
    EfficientNet-B0 inflated to 3D, uses SiLU activations, squeeze-and-excitation,
    and asymmetric strides to preserve the thin Z=14 axis throughout
    """
    # (num_blocks, in_ch, out_ch, expand_ratio, kernel_size, stride)
    _STAGES = [
        (1,  32,  16, 1, 3, 1),
        (2,  16,  24, 6, 3, 2),
        (2,  24,  40, 6, 5, 2),
        (3,  40,  80, 6, 3, 2),
        (3,  80, 112, 6, 5, 1),
        (4, 112, 192, 6, 5, 2),
        (1, 192, 320, 6, 3, 1),
    ]

    def __init__(self, num_classes=COMP_CONFIG.num_classes, input_shape=(46, 54, 14), dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv3d(1, 32, kernel_size=(3, 3, 3), stride=(2, 2, 1), padding=(1, 1, 1), bias=False),
            nn.BatchNorm3d(32),
            nn.SiLU(inplace=True),
        )
        # MBConv stages
        self.stages = nn.Sequential(*[
            self._make_stage(n, ic, oc, ex, k, s)
            for n, ic, oc, ex, k, s in self._STAGES
        ])
        self.head_conv = nn.Sequential(
            nn.Conv3d(320, 1280, 1, bias=False),
            nn.BatchNorm3d(1280),
            nn.SiLU(inplace=True),
        )
        self.avgpool = nn.AdaptiveAvgPool3d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(1280, num_classes),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm3d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                nn.init.zeros_(m.bias)

    def _make_stage(self, num_blocks, in_ch, out_ch, expand_ratio, kernel_size, stride):
        blocks = [MBConv3D(in_ch, out_ch, expand_ratio, kernel_size, stride)]
        for _ in range(1, num_blocks):
            blocks.append(MBConv3D(out_ch, out_ch, expand_ratio, kernel_size, 1))
        return nn.Sequential(*blocks)

    def forward(self, x, classify=True):
        x = self.stem(x)
        x = self.stages(x)
        x = self.head_conv(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


# === MedicalNet3D ===
class MedicalNet3D(nn.Module):
    def __init__(
        self, input_shape=(46, 54, 14), num_classes=COMP_CONFIG.num_classes, pretrained=True,
        pth_path='pre-trained/MedicalNet/resnet_10_23dataset.pth'
    ):
        super().__init__()
        # build ResNet-10 backbone (1 BasicBlock3D per layer)
        self._in_ch  = 64
        self.conv1   = nn.Conv3d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1     = nn.BatchNorm3d(64)
        self.relu    = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool3d(kernel_size=3, stride=2, padding=1)
        self.layer1  = self._make_layer(64,  stride=1)
        self.layer2  = self._make_layer(128, stride=2)
        self.layer3  = self._make_layer(256, stride=2)
        self.layer4  = self._make_layer(512, stride=2)
        self.avgpool = nn.AdaptiveAvgPool3d(1)
        self.head = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(512, num_classes)
        )
        if pretrained:
            print(f"Loading MedicalNet ResNet-10 weights from: {pth_path} ...")
            ckpt = torch.load(pth_path, map_location='cpu')
            # checkpoint may store weights under a 'state_dict' key
            sd = ckpt.get('state_dict', ckpt) if isinstance(ckpt, dict) else ckpt
            # strip the 'module.' prefix added by DataParallel training
            sd = {k.replace('module.', '', 1): v for k, v in sd.items()}
            # strict=False: the original checkpoint has no classifier head
            missing, unexpected = self.load_state_dict(sd, strict=False)
            print(f"Weights loaded.  Missing: {len(missing)}  Unexpected: {len(unexpected)}")
            if missing:
                print(f"Missing keys (expected, these are the new head): {missing}")
        else:
            print("Pretrained=False: weights randomly initialised")

    def _make_layer(self, out_channels, stride):
        downsample = None
        if stride != 1 or self._in_ch != out_channels:
            downsample = nn.Sequential(
                nn.Conv3d(self._in_ch, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm3d(out_channels),
            )
        layer = nn.Sequential(BasicBlock3D(self._in_ch, out_channels, stride, downsample))
        self._in_ch = out_channels
        return layer

    def forward(self, x, classify=True):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.head(x)


# I3D  (Inflated 3D ConvNet: inception-v1 inflated to 3D)
class I3D(nn.Module):
    def __init__(self, num_classes=COMP_CONFIG.num_classes, input_shape=(46, 54, 14), pretrained=True, dropout=0.5):
        super().__init__()
        print('Loading I3D (i3d_r50) from pytorchvideo (Kinetics-400 pretrained)...')
        backbone = torch.hub.load(
            'facebookresearch/pytorchvideo:main', 'i3d_r50',
            pretrained=pretrained, verbose=False,
        )
        # adapt stem conv: 3-ch -> 1-ch (average RGB weights)
        stem_conv = backbone.blocks[0].conv
        old_w = stem_conv.weight.data
        stem_conv.weight = nn.Parameter(old_w.mean(dim=1, keepdim=True))
        print('Stem conv adapted: 3-channel -> 1-channel (weights averaged)')
        # remove classification head (blocks[5]), keep backbone blocks 0-4
        self.backbone = nn.Sequential(*list(backbone.blocks[:-1]))
        # classification head
        self.avgpool = nn.AdaptiveAvgPool3d(1)
        self.head = nn.Sequential(
            nn.Dropout(p=0.7),
            nn.Linear(2048, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.4),
            nn.Linear(128, num_classes),
        )
        # initialise the new head layers
        for layer in self.head:
            if isinstance(layer, nn.Linear):
                nn.init.normal_(layer.weight, 0, 0.01)
                nn.init.constant_(layer.bias, 0)

    def forward(self, x, classify=True):
        x = self.backbone(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.head(x)


# METRICS SAVING
def save_metrics_csv(metrics_dict, phase, model_name, results_dir):
    df = pd.DataFrame(metrics_dict)
    csv_path = os.path.join(results_dir, f"{model_name}_{phase}_metrics.csv")
    df.to_csv(csv_path, index=False)
    print(f"{phase.capitalize()} metrics saved: {csv_path}")


# UNIVERSAL MODEL TRAINER
class UniversalModelTrainer:
    def __init__(self, model, model_name, config):
        self.model = model.to(config.device)
        self.model_name = model_name
        self.config = config
        self.device = config.device
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
        self.scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(self.optimizer, T_0=5, T_mult=2, eta_min=1e-6)
        self.train_losses = []
        self.val_losses = []
        self.train_accuracies = []
        self.val_accuracies = []
        self.best_val_acc = 0.0
        self.best_epoch = 0
        self.best_train_acc = 0.0
        self.best_train_loss = 0.0
        self.best_val_loss = float('inf')
        self.epoch_metrics = {
            'epoch': [],
            'train_loss': [],
            'train_acc': [],
            'val_loss': [],
            'val_acc': []
        }

    def train_epoch(self, train_loader):
        self.model.train()
        total_loss = 0
        correct = 0
        total = 0
        pbar = tqdm(train_loader, desc='Training', leave=False)
        for volumes, medical, labels, _ in pbar:
            volumes = volumes.to(self.device)
            labels = labels.to(self.device)
            self.optimizer.zero_grad()
            outputs = self.model(volumes)
            loss = self.criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=5.0)
            self.optimizer.step()
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100. * correct / total:.2f}%'})
        avg_loss = total_loss / len(train_loader)
        accuracy = 100. * correct / total
        return avg_loss, accuracy

    def validate(self, val_loader):
        self.model.eval()
        total_loss = 0
        correct = 0
        total = 0
        all_preds = []
        all_labels = []
        with torch.no_grad():
            pbar = tqdm(val_loader, desc='Validation', leave=False)
            for volumes, medical, labels, _ in pbar:
                volumes = volumes.to(self.device)
                labels = labels.to(self.device)
                outputs = self.model(volumes)
                loss = self.criterion(outputs, labels)
                total_loss += loss.item()
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100. * correct / total:.2f}%'})
        avg_loss = total_loss / len(val_loader)
        accuracy = 100. * correct / total
        return avg_loss, accuracy, all_preds, all_labels

    def train(self, train_loader, val_loader):
        patience_counter = 0
        epoch_pbar = tqdm(range(self.config.num_epochs), desc=f'Training {self.model_name}')
        for epoch in epoch_pbar:
            train_loss, train_acc = self.train_epoch(train_loader)
            val_loss, val_acc, _, _ = self.validate(val_loader)
            self.train_losses.append(train_loss)
            self.val_losses.append(val_loss)
            self.train_accuracies.append(train_acc)
            self.val_accuracies.append(val_acc)
            self.epoch_metrics['epoch'].append(epoch + 1)
            self.epoch_metrics['train_loss'].append(train_loss)
            self.epoch_metrics['train_acc'].append(train_acc)
            self.epoch_metrics['val_loss'].append(val_loss)
            self.epoch_metrics['val_acc'].append(val_acc)
            self.scheduler.step()
            epoch_pbar.set_postfix({
                'train_loss': f'{train_loss:.4f}',
                'train_acc': f'{train_acc:.2f}%',
                'val_loss': f'{val_loss:.4f}',
                'val_acc': f'{val_acc:.2f}%'
            })
            print(f"Epoch {epoch + 1}/{self.config.num_epochs} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

            if val_acc > self.best_val_acc:
                self.best_val_acc = val_acc
                self.best_epoch = epoch + 1
                self.best_train_acc = train_acc
                self.best_train_loss = train_loss
                self.best_val_loss = val_loss
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'best_val_acc': self.best_val_acc
                }, os.path.join(self.config.models_dir, f'{self.model_name}_best.pth'))

                print(f"New best model saved (Epoch {self.best_epoch}, Val Acc: {self.best_val_acc:.2f}%)")
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= self.config.patience:
                    print(f"\nEarly stopping triggered at epoch {epoch + 1}")
                    break

        # restore best model
        best_checkpoint = torch.load(os.path.join(self.config.models_dir, f'{self.model_name}_best.pth'))
        self.model.load_state_dict(best_checkpoint['model_state_dict'])
        print(f"\nRestored best model from epoch {self.best_epoch}")
        save_metrics_csv(self.epoch_metrics, 'train_val', self.model_name, self.config.results_dir)
        return {
            'best_val_acc': self.best_val_acc,
            'best_train_acc': self.best_train_acc,
            'best_train_loss': self.best_train_loss,
            'best_val_loss': self.best_val_loss,
            'best_epoch': self.best_epoch
        }

    def test(self, test_loader):
        self.model.eval()
        all_preds = []
        all_labels = []
        total_loss = 0.0
        num_batches = 0
        with torch.no_grad():
            for volumes, medical, labels, _ in test_loader:
                volumes = volumes.to(self.device)
                labels = labels.to(self.device)
                outputs = self.model(volumes)
                total_loss += self.criterion(outputs, labels).item()
                num_batches += 1
                _, predicted = outputs.max(1)
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        test_loss = total_loss / num_batches if num_batches > 0 else 0.0
        accuracy = accuracy_score(all_labels, all_preds) * 100
        precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted', zero_division=0)

        test_results_df = pd.DataFrame({'y_true': all_labels, 'y_pred': all_preds})
        csv_path = os.path.join(self.config.results_dir, f"{self.model_name}_test_predictions.csv")
        test_results_df.to_csv(csv_path, index=False)
        print(f"Test predictions saved: {csv_path}")
        return {
            'accuracy': accuracy,
            'loss': test_loss,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'predictions': all_preds,
            'labels': all_labels,
            'confusion_matrix': confusion_matrix(all_labels, all_preds)
        }

    def plot_training_history(self):
        fig, ax1 = plt.subplots(figsize=(12, 5))
        epochs = range(1, len(self.train_losses) + 1)
        # loss on left y-axis
        ax1.plot(epochs, self.train_losses, 'b-', label='Train Loss', linewidth=2)
        ax1.plot(epochs, self.val_losses, 'r-', label='Val Loss', linewidth=2)
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss', color='black')
        ax1.tick_params(axis='y')
        ax1.grid(True, alpha=0.3)
        # accuracy on right y-axis
        ax2 = ax1.twinx()
        ax2.plot(epochs, self.train_accuracies, 'b--', label='Train Acc', linewidth=2)
        ax2.plot(epochs, self.val_accuracies, 'r--', label='Val Acc', linewidth=2)
        ax2.set_ylabel('Accuracy (%)', color='black')
        ax2.tick_params(axis='y')
        # combined legend
        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')
        plt.title(f'{self.model_name} - Loss & Accuracy')
        plt.tight_layout()

        save_path = os.path.join(self.config.graphs_dir, f'{self.model_name}_training.png')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"Training curves saved: {save_path}")


# VISUALIZATION FUNCTIONS
def plot_confusion_matrix(cm, model_name):
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=[f'Stage {i}' for i in range(5)],
        yticklabels=[f'Stage {i}' for i in range(5)]
    )
    plt.title(f'{model_name} - Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()

    save_path = os.path.join(COMP_CONFIG.graphs_dir, f'{model_name}_confusion_matrix.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Confusion matrix saved: {save_path}")


def generate_comparison_report(results):
    comparison_data = []
    for model_name, result in results.items():
        if result is not None:
            comparison_data.append({
                'Model': model_name,
                'Best Epoch': result['best_epoch'],
                'Train Acc (%)': f"{result['best_train_acc']:.2f}",
                'Train Loss': f"{result['best_train_loss']:.4f}",
                'Val Acc (%)': f"{result['best_val_acc']:.2f}",
                'Val Loss': f"{result['best_val_loss']:.4f}",
                'Test Acc (%)': f"{result['test_accuracy']:.2f}",
                'Test Loss': f"{result['test_loss']:.4f}",
                'Test Precision': f"{result['test_precision']:.4f}",
                'Test Recall': f"{result['test_recall']:.4f}",
                'Test F1': f"{result['test_f1']:.4f}",
                'Execution Time (sec)': f"{result['execution_time_sec']:.2f}",
                'Execution Time (min)': f"{result['execution_time_sec']/60:.2f}",
                'Avg CPU (%)': f"{result['avg_cpu_percent']:.2f}",
                'Peak Memory (MB)': f"{result['peak_memory_mb']:.2f}",
                'Memory Used (MB)': f"{result['memory_used_mb']:.2f}"
            })

    if not comparison_data:
        print("\nNo successful model results to generate comparison report")
        return
    df = pd.DataFrame(comparison_data)
    csv_path = os.path.join(COMP_CONFIG.results_dir, 'model_comparison.csv')
    df.to_csv(csv_path, index=False)
    print(f"\nResults saved: {csv_path}")

    # generate visualization
    fig, ax = plt.subplots(figsize=(12, 6))
    models_list = df['Model'].tolist()
    test_accs = [float(x) for x in df['Test Acc (%)'].tolist()]
    colors = [plt.cm.tab10(i) for i in range(len(models_list))]
    bars = ax.bar(models_list, test_accs, color=colors[:len(models_list)])
    ax.set_ylabel('Test Accuracy (%)')
    ax.set_title('Model Performance Comparison')
    ax.set_ylim([0, 100])
    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold'
        )
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()

    save_path = os.path.join(COMP_CONFIG.graphs_dir, 'model_comparison.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Comparison chart saved: {save_path}\n")


# BATCH TRAINING PIPELINE
def train_and_test_all_models(train_loader, val_loader, test_loader):
    models_to_run = {
        'MobileNet3D': MobileNet_3D(
            input_shape=(46, 54, 14), num_classes=COMP_CONFIG.num_classes
        ),
        'ResNet3D-18': ResNet3D_18(
            input_shape=(46, 54, 14), num_classes=COMP_CONFIG.num_classes
        ),
        'VGG3D-16':    VGG3D_16(
            input_shape=(46, 54, 14), num_classes=COMP_CONFIG.num_classes
        ),
        'EfficientNet3D-B0': EfficientNet3D_B0(
            num_classes=COMP_CONFIG.num_classes, input_shape=(46, 54, 14), dropout=0.3
        ),
        'MedicalNet3D': MedicalNet3D(
            input_shape=(46, 54, 14), num_classes=COMP_CONFIG.num_classes, pretrained=True,
            pth_path='/kaggle/input/models/tolbaadel/medicalnet/pytorch/resnet_10_23datasets/1/resnet_10_23dataset.pth',
        ),
        'I3D': I3D(
            num_classes=COMP_CONFIG.num_classes, input_shape=(46, 54, 14), pretrained=True, dropout=0.5
        )
    }
    results = {}

    for model_name, model in models_to_run.items():
        print(f"\n{'#'*80}\n# {model_name}\n{'#'*80}")

        # clear GPU cache before training
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

        try:
            # start resource monitoring
            process = psutil.Process()
            start_time = time.time()
            process.cpu_percent()
            start_memory = process.memory_info().rss / 1024**2  # MB

            trainer = UniversalModelTrainer(model, model_name, COMP_CONFIG)
            train_results = trainer.train(train_loader, val_loader)
            test_results = trainer.test(test_loader)
            trainer.plot_training_history()

            # end resource monitoring
            end_time = time.time()
            avg_cpu_percent = process.cpu_percent()
            end_memory = process.memory_info().rss / 1024**2  # MB
            execution_time = end_time - start_time
            memory_used = max(0, end_memory - start_memory)
            peak_memory = end_memory

            results[model_name] = {
                'best_val_acc': train_results['best_val_acc'],
                'best_train_acc': train_results['best_train_acc'],
                'best_train_loss': train_results['best_train_loss'],
                'best_val_loss': train_results['best_val_loss'],
                'best_epoch': train_results['best_epoch'],
                'test_accuracy': test_results['accuracy'],
                'test_loss': test_results['loss'],
                'test_precision': test_results['precision'],
                'test_recall': test_results['recall'],
                'test_f1': test_results['f1'],
                'confusion_matrix': test_results['confusion_matrix'],
                'execution_time_sec': execution_time,
                'avg_cpu_percent': avg_cpu_percent,
                'peak_memory_mb': peak_memory,
                'memory_used_mb': memory_used
            }
            print(f"\nResource Usage:")
            print(f"  Execution Time: {execution_time:.2f} seconds ({execution_time/60:.2f} minutes)")
            print(f"  Avg CPU Usage: {avg_cpu_percent:.2f}%")
            print(f"  Peak Memory: {peak_memory:.2f} MB")
            print(f"  Memory Used: {memory_used:.2f} MB")
            plot_confusion_matrix(test_results['confusion_matrix'], model_name)
        except Exception as e:
            print(f"{model_name} failed: {e}")
            import traceback
            traceback.print_exc()
            results[model_name] = None

    generate_comparison_report(results)
    return results



# MAIN EXECUTION
print("Starting comprehensive model training and testing...")
print(f"Training samples: {len(train_loader.dataset)}")
print(f"Validation samples: {len(val_loader.dataset)}")
print(f"Test samples: {len(test_loader.dataset)}\n")

results = train_and_test_all_models(train_loader, val_loader, test_loader)

print("\n================ FINAL RESULTS SUMMARY ================")
successful_models = 0
failed_models = 0
for model_name, result in results.items():
    if result is not None:
        print(f"\nModel: {model_name}")
        print(f"  Best Epoch: {result['best_epoch']}")
        print(f"  Best Train Accuracy: {result['best_train_acc']:.2f}%")
        print(f"  Best Validation Accuracy: {result['best_val_acc']:.2f}%")
        print(f"  Test Accuracy: {result['test_accuracy']:.2f}%")
        successful_models += 1
    else:
        print(f"\nModel: {model_name} - Training/Testing Failed")
        failed_models += 1
print(f"\nTraining complete: {successful_models} model(s) successful, {failed_models} model(s) failed.")
print("\n======================================================")